In [ ]:
# MINING FREQUENT WEIGHTED PATTERNS OVER DATA STREAMS
# 1. Format(login, clear and complete)
# 2. Define the problem (input, output in terms of Python3 data structures) (clear, logic and complete)
# 3. Implement (in Python3 and accompanied with documentation) the algorithms to solve the problem without importing libraries, such as numpy(complete)
# 4. Measure the running time for benchmark datasets(clear, login and complete)
# 5. Question-Answering Session(clear, login and complete)





# Định nghĩa vấn đề:
# Cho một luồng dữ liệu của các mẫu có trọng số, nhiệm vụ là khai thác các mẫu
# thường xuyên và trọng số tương ứng từ luồng.

# Đầu vào:
# - Một luồng dữ liệu bao gồm các bộ (mẫu, trọng số), trong đó mẫu là một chuỗi
#   và trọng số là một giá trị số.

# Đầu ra:
# - Danh sách các mẫu thường xuyên cùng với trọng số tương ứng.

# Ví dụ:
# stream = [("A", 3), ("B", 2), ("A", 5), ("C", 3), ("B", 4), ("A", 2)]
# min_support = 2

# -> [("A", 10), ("B", 6)]





# Đánh giá hiệu suất:

# # [Cung cấp chi tiết về bộ dữ liệu đánh giá và cách chạy đánh giá]

# # Ví dụ:
# import time
# from frequent_patterns_miner import find_frequent_weighted_patterns

# start_time = time.time()
# # [Cuộc gọi hàm của bạn để đánh giá hiệu suất ở đây]
# end_time = time.time()

# print(f"Thời gian thực thi: {end_time - start_time} giây")




In [ ]:
import pandas as pd
import random

# Define the number of transactions and items
num_transactions = 10
num_items = 5

# Create lists to store data
transaction_ids = range(1, num_transactions + 1)
items = []
weights = []

# Generate the transactions
for _ in range(num_transactions):
    transaction_items = random.sample(['A', 'B', 'C', 'D', 'E'], random.randint(1, num_items))
    items.append(', '.join(transaction_items))

# Generate random weights for items
item_weights = {
    'A': random.randint(1, 7),
    'B': random.randint(1, 7),
    'C': random.randint(1, 7),
    'D': random.randint(1, 7),
    'E': random.randint(1, 7)
}

# Create lists for item and weight columns
item_column = []
weight_column = []

for item, weight in item_weights.items():
    item_column.append(item)
    weight_column.append(weight)

# Create the DataFrame
df1 = pd.DataFrame({
    'TID': transaction_ids,
    'Items': items
})

df2 = pd.DataFrame({
    'Items': item_column,
    'Weights': weight_column
})

# Display the DataFrames
print("Table 1: Example of WD")
print(df1)

print("\nTable 2: The weights of items")
print(df2)

# Save DataFrames to CSV files
df1.to_csv('dataSets.csv', index=False)
df2.to_csv('itemsWeights.csv', index=False)

########################################################################################################################

# Read the transaction data from dataSets.csv
df1 = pd.read_csv('dataSets.csv')

# Read the item weights data from itemsWeights.csv
df2 = pd.read_csv('itemsWeights.csv')



item_weights = dict(zip(df2['Items'], df2['Weights']))

# Calculate transaction weighted (tw)
# Split the items in the 'Items' column and calculate the tw for each transaction
def calculate_tw(row):
    items_in_transaction = row['Items'].split(', ')
    tw = sum(item_weights[item] for item in items_in_transaction) / len(items_in_transaction)
    return tw

df1['tw'] = df1.apply(calculate_tw, axis=1)

# Create the DataFrame for tw
df_tw = df1[['TID', 'tw']]

# Calculate TTW (Total Transaction Weights)
TTW = df1['tw'].sum()

# Calculate weighted support (ws)
def calculate_ws(item):
    total_tw_with_item = df1[df1['Items'].str.contains(item)]['tw'].sum()
    ws = total_tw_with_item / TTW
    return ws

items = item_weights.keys()
ws_values = [calculate_ws(item) for item in items]

# Create the DataFrame for ws
df_ws = pd.DataFrame({'Items': items, 'ws': ws_values})
df_ws = df_ws.sort_values(by='ws', ascending=False)

# Display the DataFrames
print("\nTable example for tw:")
print(df1[['TID', 'tw']])

print("\nTTW:")
print(TTW)

print("\nTable example for ws:")
print(df_ws)

Table 1: Example of WD
   TID          Items
0    1        A, E, C
1    2           A, E
2    3     B, C, E, D
3    4        D, C, A
4    5              A
5    6        A, E, B
6    7     C, E, D, A
7    8  E, D, C, B, A
8    9  D, B, A, E, C
9   10              B

Table 2: The weights of items
  Items  Weights
0     A        1
1     B        1
2     C        2
3     D        1
4     E        3

Table example for tw:
   TID        tw
0    1  2.000000
1    2  2.000000
2    3  1.750000
3    4  1.333333
4    5  1.000000
5    6  1.666667
6    7  1.750000
7    8  1.600000
8    9  1.600000
9   10  1.000000

TTW:
15.700000000000001

Table example for ws:
  Items        ws
0     A  0.824841
4     E  0.787686
2     C  0.639066
3     D  0.511677
1     B  0.485138


In [ ]:
import pandas as pd

# Function to read the transaction data from a CSV file
def read_transaction_data(filename):
    df = pd.read_csv(filename)
    return df

# Function to read the item weights data from a CSV file
def read_item_weights(filename):
    df = pd.read_csv(filename)
    item_weights = dict(zip(df['Items'], df['Weights']))
    return item_weights

# Function to calculate transaction weighted (tw)
def calculate_tw(df, item_weights):
    def calculate_tw_row(row):
        items_in_transaction = row['Items'].split(', ')
        tw = sum(item_weights[item] for item in items_in_transaction) / len(items_in_transaction)
        return tw

    df['tw'] = df.apply(calculate_tw_row, axis=1)
    return df

# Function to implement sliding window model
def sliding_window(df, item_weights, window_size, panel_size, min_ws):
    num_transactions = len(df)

    for i in range(num_transactions - window_size + 1):
        window = df[i:i + window_size]
        window_number = i + 1

        print(f"Window {window_number}:\n{window}")

        panel = df.iloc[i + window_size:i + window_size + panel_size]
        items_in_panel = []

        for _, row in panel.iterrows():
            items_in_panel.extend(row['Items'].split(', '))

        items_in_panel = set(items_in_panel)

        ws_sum = 0
        relevant_transactions = []

        for j in range(i, i + window_size):
            transaction = df.iloc[j]
            items_in_transaction = set(transaction['Items'].split(', '))

            if items_in_panel.issubset(items_in_transaction):
                relevant_transactions.append(transaction)

        if len(relevant_transactions) == 0:
            continue

        # Calculate the TTW for the current window
        window_ttw = sum(transaction.tw for transaction in window.itertuples(index=False))

        # Calculate the weighted support for the itemset X in the panel
        tw = sum(transaction['tw'] for transaction in relevant_transactions)
        ws = tw / window_ttw

        if ws >= min_ws:
            print(f"Panel {i + window_size + 1}:\n{panel}")
            print(f"Weighted Support of items in panel (X): {ws}")

# Main function
def main():
    window_size = 5
    panel_size = 1
    min_ws = 0.3

    # Read the transaction data and item weights data from CSV files
    df = read_transaction_data('dataSets.csv')
    item_weights = read_item_weights('itemsWeights.csv')

    # Calculate transaction weighted (tw)
    df = calculate_tw(df, item_weights)
    df_tw = df[['TID', 'tw']]

    # Implement sliding window model to find FWPs
    sliding_window(df, item_weights, window_size, panel_size, min_ws)

if __name__ == "__main__":
    main()

Window 1:
   TID       Items        tw
0    1     A, E, C  2.000000
1    2        A, E  2.000000
2    3  B, C, E, D  1.750000
3    4     D, C, A  1.333333
4    5           A  1.000000
Window 2:
   TID       Items        tw
1    2        A, E  2.000000
2    3  B, C, E, D  1.750000
3    4     D, C, A  1.333333
4    5           A  1.000000
5    6     A, E, B  1.666667
Window 3:
   TID       Items        tw
2    3  B, C, E, D  1.750000
3    4     D, C, A  1.333333
4    5           A  1.000000
5    6     A, E, B  1.666667
6    7  C, E, D, A  1.750000
Window 4:
   TID          Items        tw
3    4        D, C, A  1.333333
4    5              A  1.000000
5    6        A, E, B  1.666667
6    7     C, E, D, A  1.750000
7    8  E, D, C, B, A  1.600000
Window 5:
   TID          Items        tw
4    5              A  1.000000
5    6        A, E, B  1.666667
6    7     C, E, D, A  1.750000
7    8  E, D, C, B, A  1.600000
8    9  D, B, A, E, C  1.600000
Panel 10:
   TID Items   tw
9   10     B  1.

In [ ]:
#MAKE RANDOM DATA STREAM

import pandas as pd
import random

# Tạo dữ liệu ngẫu nhiên
data = []
for _ in range(200):
    sample = random.choice(['A', 'B', 'C', 'D'])
    weight = random.randint(1, 5)
    data.append((sample, weight))

# Tạo DataFrame từ dữ liệu
df = pd.DataFrame(data, columns=['Pattern', 'Weight'])

# Lưu DataFrame vào tệp CSV
df.to_csv("input.csv", index=False)

print("Đã tạo input.csv.")


Đã tạo input.csv.


In [ ]:
# Description: Mining Frequent Weighted Patterns over Data Streams


import pandas as pd
from collections import defaultdict

def generate_candidates(prev_candidates, k):
    """Generate candidate patterns of length k."""
    candidates = set()
    for p1 in prev_candidates:
        for p2 in prev_candidates:
            if p1[:-1] == p2[:-1] and p1[-1] < p2[-1]:
                candidates.add(p1 + (p2[-1],))
    return candidates

def calculate_support(df, candidate):
    """Calculate support for a candidate pattern in the dataframe."""
    count = df[df['Pattern'].apply(lambda x: set(candidate).issubset(set(x)))]['Weight'].sum()
    return count

def apriori_algorithm(df, min_support):
    """Apriori algorithm to mine frequent weighted patterns."""
    patterns = []
    k = 1

    # Get unique items from the dataframe
    unique_items = set(item for patterns_set in df['Pattern'] for item in patterns_set)

    # Initialize frequent patterns of length 1
    frequent_patterns = {(item,) for item in unique_items if calculate_support(df, (item,)) >= min_support}

    while frequent_patterns:
        patterns.extend(frequent_patterns)
        k += 1
        candidate_patterns = generate_candidates(frequent_patterns, k)
        frequent_patterns = {candidate for candidate in candidate_patterns if calculate_support(df, candidate) >= min_support}

    return patterns

def find_frequent_weighted_patterns(df, min_support):
    """Find frequent weighted patterns from the dataframe."""
    frequent_patterns = apriori_algorithm(df, min_support)
    result = [(pattern, df[df['Pattern'].apply(lambda x: set(pattern).issubset(set(x)))]['Weight'].sum()) for pattern in frequent_patterns]
    return result

# Example usage
df = pd.read_csv("input.csv")

# Specify the minimum support threshold
min_support = 10

# Find frequent weighted patterns
frequent_weighted_patterns = find_frequent_weighted_patterns(df, min_support)

# Print the result
print("Frequent Weighted Patterns:", frequent_weighted_patterns)




Frequent Weighted Patterns: [(('C',), 146), (('D',), 195), (('B',), 137), (('A',), 149)]


In [ ]:
# FWPODS ALGORITHM

import pandas as pd
from collections import defaultdict

def generate_candidates(prev_candidates, k):
    """Generate candidate patterns of length k."""
    candidates = set()
    for p1 in prev_candidates:
        for p2 in prev_candidates:
            if p1[:-1] == p2[:-1] and p1[-1] < p2[-1]:
                candidates.add(p1 + (p2[-1],))
    return candidates

def calculate_support(df, candidate):
    """Calculate support for a candidate pattern in the dataframe."""
    count = df[df['Pattern'].apply(lambda x: set(candidate).issubset(set(x)))]['Weight'].sum()
    return count

def fpwods_algorithm(df, min_support):
    """FWPODS algorithm to mine frequent weighted patterns."""
    patterns = []
    k = 1

    # Get unique items from the dataframe
    unique_items = set(item for patterns_set in df['Pattern'] for item in patterns_set)

    # Initialize frequent patterns of length 1
    frequent_patterns = {(item,) for item in unique_items if calculate_support(df, (item,)) >= min_support}

    while frequent_patterns:
        patterns.extend(frequent_patterns)
        k += 1
        candidate_patterns = generate_candidates(frequent_patterns, k)
        frequent_patterns = {candidate for candidate in candidate_patterns if calculate_support(df, candidate) >= min_support}

    return patterns

# Read data from Excel file
df = pd.read_csv("input.csv")

# Specify the minimum support threshold
min_support = 10

# Find frequent weighted patterns using FWPODS algorithm
frequent_weighted_patterns = fpwods_algorithm(df, min_support)

# Print the result in a clear format
print("Danh sách các mẫu thường xuyên cùng với trọng số tương ứng:")
for pattern in frequent_weighted_patterns:
    print(f"Mẫu: {pattern}, Trọng số: {calculate_support(df, pattern)}")



Danh sách các mẫu thường xuyên cùng với trọng số tương ứng:
Mẫu: ('C',), Trọng số: 146
Mẫu: ('D',), Trọng số: 195
Mẫu: ('B',), Trọng số: 137
Mẫu: ('A',), Trọng số: 149


In [ ]:
# Implement (in Python3 and accompanied with documentation) the FWPODS ALGORITHMS
# to solve the problem without importing libraries, such as numpy

import csv

def calculate_support(data, candidate):
    """Calculate support for a candidate pattern in the dataset."""
    count = sum(1 for row in data if set(candidate).issubset(set(row[0])))
    return count

def generate_patterns(frequent_patterns, k):
    """Generate candidate patterns of length k from frequent patterns of length k-1."""
    patterns = set()
    for pattern1 in frequent_patterns:
        for pattern2 in frequent_patterns:
            new_pattern = tuple(sorted(set(pattern1).union(pattern2)))
            if len(new_pattern) == k and new_pattern not in patterns:
                patterns.add(new_pattern)
    return patterns

def fpwods_algorithm(data, min_support):
    """FWPODS algorithm to mine frequent weighted patterns."""
    patterns = []
    k = 1

    # Get unique items from the dataset
    unique_items = set(item for row in data for item in row[0])

    # Initialize frequent patterns of length 1
    frequent_patterns = {(item,) for item in unique_items if calculate_support(data, (item,)) >= min_support}

    while frequent_patterns:
        patterns.extend(frequent_patterns)
        k += 1
        candidate_patterns = generate_patterns(frequent_patterns, k)
        frequent_patterns = {candidate for candidate in candidate_patterns if calculate_support(data, candidate) >= min_support}

    return patterns

def print_frequent_patterns(frequent_patterns, data):
    """Print frequent patterns along with their weights."""
    print("Danh sách các mẫu thường xuyên cùng với trọng số tương ứng:")
    for pattern in frequent_patterns:
        # Truy cập mẫu từ tuple (item,) bằng cách sử dụng pattern[0]
        print(f"Mẫu: {pattern}, Trọng số: {sum(row[1] for row in data if set(pattern).issubset(set(row[0])))}")

    print("Loaded Data:", data)

def read_data_from_csv(file_path):
    """Read data from a CSV file."""
    data = []
    with open(file_path, 'r') as file:
        reader = csv.reader(file)

        # Skip the header row
        next(reader)

        for row in reader:
            # Assuming the first column contains the pattern and the second column contains the weight
            pattern = tuple(row[0].split(','))  # Convert the pattern to a tuple
            weight = int(row[1])
            data.append((pattern, weight))
    return data

# Replace 'input.csv' with the actual path to your CSV file
file_path = 'input.csv'
data = read_data_from_csv(file_path)

# Specify the minimum support threshold
min_support = 2

# Find frequent weighted patterns using FWPODS algorithm
frequent_weighted_patterns = fpwods_algorithm(data, min_support)

# Print the result in the desired format
print_frequent_patterns(frequent_weighted_patterns, data)



Danh sách các mẫu thường xuyên cùng với trọng số tương ứng:
Mẫu: ('C',), Trọng số: 146
Mẫu: ('D',), Trọng số: 195
Mẫu: ('B',), Trọng số: 137
Mẫu: ('A',), Trọng số: 149
Loaded Data: [(('C',), 1), (('D',), 5), (('A',), 2), (('D',), 2), (('A',), 4), (('D',), 1), (('C',), 5), (('A',), 4), (('B',), 2), (('C',), 4), (('D',), 5), (('B',), 3), (('C',), 4), (('D',), 2), (('C',), 3), (('C',), 4), (('C',), 4), (('B',), 3), (('A',), 4), (('A',), 1), (('D',), 4), (('C',), 4), (('B',), 3), (('C',), 2), (('B',), 5), (('A',), 4), (('B',), 3), (('D',), 3), (('B',), 3), (('A',), 5), (('B',), 2), (('A',), 3), (('B',), 2), (('A',), 3), (('A',), 1), (('B',), 3), (('D',), 5), (('A',), 5), (('D',), 1), (('D',), 4), (('A',), 3), (('C',), 2), (('B',), 3), (('B',), 2), (('B',), 5), (('D',), 5), (('B',), 4), (('A',), 4), (('D',), 4), (('A',), 5), (('D',), 1), (('B',), 4), (('D',), 4), (('C',), 5), (('B',), 4), (('C',), 1), (('A',), 2), (('C',), 5), (('C',), 1), (('B',), 1), (('B',), 2), (('B',), 3), (('B',), 3),

In [ ]:
# Measure the running time for benchmark datasets

import csv
import time

def extract_numeric_value(value):
    """Extract numeric value from a string."""
    numeric_chars = [char for char in value if char.isdigit() or char in ('.', '-')]
    numeric_str = ''.join(numeric_chars)
    try:
        return int(numeric_str) if '.' not in numeric_str else float(numeric_str)
    except ValueError:
        return 0  # Default value if conversion fails

def read_data_from_csv(file_path):
    """Read data from a CSV file and return a list of tuples (Pattern, Weight)."""
    data = []
    with open(file_path, 'r') as file:
        reader = csv.reader(file)

        # Skip the header row
        next(reader)

        for row in reader:
            # Assuming the first column contains the pattern and the second column contains the weight
            pattern = tuple(row[0].split(','))  # Convert the pattern to a tuple
            weight = extract_numeric_value(row[1])
            data.append((pattern, weight))
    return data

def fpwods_algorithm(data, min_support):
    # Implement your FWPODS algorithm here
    pass

def print_frequent_patterns(frequent_patterns, data):
    # Implement your printing logic here
    pass

# Replace 'input.csv' with the actual path to your CSV file
file_path = 'input.csv'
data = read_data_from_csv(file_path)

# Specify the minimum support threshold
min_support = 2

# Record the start time
start_time = time.time()

# Find frequent weighted patterns using FWPODS algorithm
frequent_weighted_patterns = fpwods_algorithm(data, min_support)

# Record the end time
end_time = time.time()

# Calculate the running time
running_time = end_time - start_time

# Print the result in the desired format
print_frequent_patterns(frequent_weighted_patterns, data)

# Print the running time
print(f"Running time: {running_time} seconds")


Running time: 8.702278137207031e-05 seconds


In [ ]:
# SWN-Tree

In [ ]:
# Maintaining SWN-Tree Over Data Stream

In [ ]:
# A. Mô hình cửa sổ trượt để khai thác FWPS qua luồng dữ liệu

# Giả sử WD là cơ sở dữ liệu có trọng số. Cơ sở dữ liệu này chứa một tập hợp các giao dịch T={t1 , t2 ,… , tm} . Giả sử I={i1 , i2,…,in} là tập hợp các mục và W={w1 , w2,…,wn} là tập hợp các trọng số tương ứng của các mục trong WD. Mỗi ti bao gồm một số mục và I là tập hợp con của ti .

# Cơ sở dữ liệu có trọng số mẫu được trình bày trong Table 1 và 2. Ví dụ này được sử dụng trong suốt nghiên cứu này.

# Table 1 mô tả tập dữ liệu bao gồm bộ 6 giao dịch T={t1,…,t6} . Trong các giao dịch này có 5 mục {A , B , C , D , E }. Table 2 trình bày trọng lượng của các hạng mục này.

# AN EXAMPLE WD
#
# TID					Items
# 1					A, C, D, F
# 2					A, B, C
# 3					C, D, F, G
# 4					A, B, C, D, G
# 5					A, B, C, D, F
# 6					B, C, E, F



# A, B, D, F
# A, B, C
# C, D, F, G
# A, B, C, D, G
# A, B, C, D, F
# B, C, E, F


# THE WEIGHTS OF ITEMS
# ITEMS				WEIGHTS
# A					0.5
# B					0.3
# C					0.7
# D					0.4
# E					0.8
# F					0.2
# G					0.9

# Định nghĩa 1:
# Trọng số giao dịch của tk∈T ký hiệu là tw(tk) được xác định theo phương trình sau:
# Tw(tk) = (Σij ∈ tk. Wj )/ |tk|

# Nói cách khác, tw(tk) là giá trị trung bình của tất cả các giá trị trọng số của các mục thuộc tk .

# Định nghĩa 2:Tổng trọng số giao dịch của cơ sở dữ liệu có trọng số được ký hiệu là TTW là tổng của tất cả các trọng số giao dịch và được xác định như sau.
# TTW = Σtk∈T (TW(tk)

# trong đó T là tập hợp tất cả các giao dịch trong WD.


# Định nghĩa 3: Cho một tập mục X . Hỗ trợ có trọng số của X ký hiệu là ws(X ) được tính theo phương trình sau.

# ws(X) = Σtk∈t(X) (tw(tk))/TTW
# trong đó t(X) là tập hợp các giao dịch chứa X .

# Ví dụ 1:Từ Table 1 và 2 và Định nghĩa 1, chúng tôi xác định trọng số giao dịch của t2 như sau: tw(t2)=(0.5+0.3+0.7)/3=0.5 .
# Tương tự, chúng tôi xác định trọng số giao dịch của tất cả các giao dịch còn lại trong cơ sở dữ liệu có trọng số ví dụ.
# Kết quả được trình bày trong Table 3. Tổng trọng số giao dịch của cơ sở dữ liệu có trọng số là 2,98, như được hiển thị
# ở dòng cuối cùng trong Table 3.





# The Transactional Weights in the Example Weighted Dataset
# TID					TRANSACTION WEIGHTS
# 1					0.45
# 2					0.50
# 3					0.55
# 4					0.56
# 5					0.42
# 6					0.50
# TTW				2.98

# Dựa trên Table 1 và 3 và Định nghĩa 2, chúng tôi đã xác định độ hỗ trợ có trọng số của tập mục BC như sau.
# Vì BC xuất hiện trong bốn giao dịch {2, 4, 5, 6} nên ws(BC)=(0.5+0.56+0.42+0.5)/2.98≈0.66 . Chúng tôi có các
# mục đã được sắp xếp theo mức hỗ trợ có trọng số giảm dần trong Table 4.


# The Weighted Supports of Items
# ITEMS				WS VALUES
# C					1
# B					0.664
# D					0.664
# A					0.648
# F					0.644
# G					0.372
# E					0.168

# Trong mô Fig cửa sổ trượt, một Table bao gồm một số giao dịch và có nhiều Table trong một cửa sổ.
# Ví dụ: hãy xem xét kích thước cửa sổ gồm 5 giao dịch và kích thước Table điều khiển là 1 giao dịch
# cho ví dụ WD trong Table 1. Fig 1 trình bày hai cửa sổ: cửa sổ 1 (phía bên trái) và cửa sổ 2 (phía bên phải).
# Cửa sổ 1 bao gồm 5 giao dịch đầu tiên (T1 đến T5). Cửa sổ 2 có được bằng cách trượt cửa sổ 1 về phía trước
# bằng 1 giao dịch. Bước này kết thúc bằng việc thêm T6 vào và xóa T1 khỏi cửa sổ 1 để tạo cửa sổ 2.


# The sliding-window model for the example dataset with a window size of 5 and panel size of 1.

# TID					Items
# 1					A, B, D, F
# 2					A, B, C
# 3					C, D, F, G
# 4					A, B, C, D, G
# 5					A, B, C, D, F
# 6					B, C, E, F



# The sliding-window model for the example dataset with a window size of 5 and panel size of 1.

# TID			Items
# {
# 1			A, C, D, F
# 2			A, B, C
# 3			C, D, F, G
# 4			A, B, C, D, G
# 5			A, B, C, D, F
# }THIS IS WINDOW 1

# 6			B, C, E, F


# TID			Items
# 1			A, C, D, F

# {
# 2			A, B, C
# 3			C, D, F, G
# 4			A, B, C, D, G
# 5			A, B, C, D, F
# 6			B, C, E, F
# }THIS IS WINDOW 2


# Vấn đề được xem xét trong nghiên cứu này là khai thác FWP trên các luồng dữ liệu và do đó tìm ra tất cả
# các mẫu có trọng số sao cho hỗ trợ có trọng số của nó đáp ứng ngưỡng hỗ trợ có trọng số tối thiểu do người
# dùng xác định (min_ws), tức là FWPs={X⊆I|ws(X) ≥min__ws} trong mỗi cửa sổ.
